walmart sales csv contains these features:
Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment

the target label is Weekly_Sales. 
i have used these features to train the model:
Store,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment

there are 45 different stores. 

installing and importing numpy and pandas:

In [2]:
%pip install numpy pandas 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 3.7 MB/s eta 0:00:00 0:00:01m

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd

loading the dataset from the csv file:

In [4]:
df=pd.read_csv("walmart_sales.csv")
print(df.head())
df.describe()

   Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0      1  05-02-2010    1643690.90             0        42.31       2.572   
1      1  12-02-2010    1641957.44             1        38.51       2.548   
2      1  19-02-2010    1611968.17             0        39.93       2.514   
3      1  26-02-2010    1409727.59             0        46.63       2.561   
4      1  05-03-2010    1554806.68             0        46.50       2.625   

          CPI  Unemployment  
0  211.096358         8.106  
1  211.242170         8.106  
2  211.289143         8.106  
3  211.319643         8.106  
4  211.350143         8.106  


,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


this creates dataframe for 45 COLUMNS, one for each store. if, lets say, the store no is 16, then the column 16 will have the binary value 1 and rest of it 0. 

this is ONE-HOT ENCODING:

In [ ]:
store_encoded = pd.get_dummies(df['Store'], prefix='Store').astype(int)
print(store_encoded)

we will concatenate these 45 new columns with the rest of the features:

and then create numpy arrays for X as input values and y as target labels. 

In [ ]:
X_df = pd.concat([
    df[["Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment"]],
    store_encoded
], axis=1)

X = np.array(X_df.values)
y = np.array(df["Weekly_Sales"].values)

In [ ]:
print(X_df.head())
print(X.shape)
print(y.shape)

calculating the different statistical terms like mean and standard deviation to normalise the values of X and y. 

In [ ]:
mu=np.mean(X, axis=0)
sigma=np.std(X, axis=0)

X_norm=(X-mu)/sigma

y_mu = np.mean(y)
y_sigma = np.std(y)

y_norm = (y - y_mu) / y_sigma

In [ ]:
print(X_norm.shape)

i need scikit learn to divide the data into train set and test set!

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

this is splitting our array into 90% training set and 10% test set. 

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X_norm,
    y_norm,
    test_size=0.10,
    random_state=42,
    shuffle=True

)

In [ ]:
print(X_train.shape, y_train.shape)


now my training data and testing data is ready. 

In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf

print(tf.__version__)

In [ ]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense

we will have four layers: three hidden with relu activation and one output with default or linear activation. 
no of units in layer 1 are 16, followed by 8, 4 and 1 in the following layers respectively. 

In [ ]:
model = Sequential([
    Dense(16, activation='relu', input_shape=(50,)),
    Dense(8, activation='relu'),
    Dense(4, activation='relu'),
    Dense(1, activation='linear' )
])

model.compile(optimizer='adam',
              loss='mse'
              )
model.fit(X_train,y_train, epochs=100)

i had scaled the y_train and y_test as well. so thats why i need to rescale all four things ie y_train, y_test and their predicted value y_train_pred and y_test_pred to 
y_train_rescaled, y_test_rescaled, y_train_pred_rescaled, y_test_pred_rescaled

In [ ]:
y_test_pred = model.predict(X_test)
y_train_pred =model.predict(X_train)

y_test_pred= y_test_pred.reshape(-1)
y_train_pred = y_train_pred.reshape(-1)

In [ ]:
y_test_pred_rescaled=(y_test_pred*y_sigma)+y_mu
y_test_rescaled=(y_test*y_sigma)+y_mu

y_train_pred_rescaled=(y_train_pred*y_sigma)+y_mu
y_train_rescaled=(y_train*y_sigma)+y_mu

In [ ]:
print(f"predicted value:{y_test_pred_rescaled[10]}")
print(f"actual value: {y_test_rescaled[10]}")

print(f"predicted value:{y_train_pred_rescaled[11]}")
print(f"actual value:{y_train_rescaled[11]}")

calculating the mean absoulte error percentage for the training set as well as test set:

In [ ]:
p_trainerr = abs(( y_train_pred_rescaled- y_train_rescaled) / y_train_rescaled) * 100
p_testerr = abs(( y_test_pred_rescaled- y_test_rescaled) / y_test_rescaled) * 100

In [ ]:
trainerr_per=np.mean(p_trainerr)
testerr_per=np.mean(p_testerr)
print(f"Training error is {trainerr_per} and testing error is {testerr_per}")

this model is giving training error as 7.70% and test set as 7.83%. we can say that theres no overfitting here. 